# Societe Generale Rates

Momentum strategies for rates: Bridging the Gap between statistics and option theory

In [1]:
import os
import numpy as np
import pandas as pd
from   scipy.stats import norm

In [2]:
write_path = os.getcwd()
res_path   = os.path.abspath(os.path.join(write_path, ".."))
repo_path  = os.path.abspath(os.path.join(res_path, ".."))
data_path  = os.path.join(repo_path, "data")
fut_path   = os.path.join(data_path, "FuturesData")

# Momentum strategies for rates: Bridging the Gap between statistics and option theory

First start by getting the universe of fixed income and STIR products. The author use all the rates and STIR futures available at the time. 

In [3]:
guide_path      = os.path.join(data_path, "TickerGuide.xlsx")
df_ticker_guide = (pd
    .read_excel(io = guide_path, sheet_name = "fut_guide")
    .loc[lambda x: x.group.isin(["fixed_income", "stir"])]
    [["Tmp", "Root Contract", "Front", "group"]]
    .rename(
        columns = {
            "Tmp"          : "name",
            "Root Contract": "root",
            "Front"        : "front"})
    .assign(ticker = lambda x: x.front.str.replace(" 1", "1").str.lower().str.replace(" ", "_")))

ticker_dict = (df_ticker_guide
    .set_index("ticker")
    .group
    .to_dict())

tickers = list(ticker_dict.keys())

In [4]:
px_path = os.path.join(fut_path, "PrepFuturesPX.parquet")
df_px   = (pd
    .read_parquet(path = px_path, engine = "pyarrow")
    .loc[lambda x: x.ticker.isin(tickers)]
    [["date", "ticker", "adj_val"]]
    .assign(group = lambda x: x.ticker.map(ticker_dict))
    .dropna()
    .rename(columns = {"adj_val": "px"}))

The author defines two types of signals for trend following, <br> 
one called past return indicator
\begin{equation}
\mathbf{Trend} = \frac{1}{T-t} \left(\ln(S_T) - \ln(S_t) \right)
\tag{1}
\end{equation}
The other is called regression line
\begin{equation}
\mathbf{Trend} = \frac{\textrm{cov}(ln(S_s), s)}{\textrm{Var}(s)} / s
\tag{2}
\end{equation}

The author uses a trend window of 100 days, and then uses a z-score window, but doesn't state what the window of the z-score is. In this case we'll start with full-sample in-sample

In [5]:
def _get_past_rtn(df: pd.DataFrame, window: int) -> pd.DataFrame: 

    df_out =(df
        .set_index("date")
        .sort_index()
        .assign(
            log_px  = lambda x: np.log(x.px),
            signal  = lambda x: 1 / (window) * (x.log_px.diff(window)),
            z_score = lambda x: (x.signal - x.signal.mean()) / x.signal.std(),
            prob    = lambda x: norm.cdf(x.z_score))
        .assign(
            lag_zscore = lambda x: x.z_score.shift(),
            lag_prob   = lambda x: x.prob.shift())
        .dropna())

    return df_out

window = 100

df_past_signal = (df_px
    .groupby("ticker")
    .apply(_get_past_rtn, window)
    .reset_index())

In [6]:
def _get_regression_signal(df: pd.DataFrame, window: int = 100) -> pd.DataFrame: 

    df_out = (df
        .sort_values("date")
        .reset_index(drop = True)
        .reset_index()
        .rename(columns = {"index": "x"})
        .assign(
            log_px = lambda x: np.log(x.px),
            x      = lambda x: x.x + 1,
            signal = lambda x: x.log_px.rolling(window = window).cov(x.x) / x.log_px.rolling(window = window).var())
        .set_index("date"))

    return df_out

df_regress_signal = (df_px
    .groupby("ticker")
    .apply(_get_regression_signal, window)
    .reset_index()
    .drop(columns = ["x"]))

There is a relationship between the z-score of the past return estimator and a relationship between options theory. With a generic geometric brownian motion 
\begin{align}
dS_t = \mu S_t d_t + \sigma S_t dB_t
\end{align}
Then consider the prices are $\ln(S_t)$ and using this transformation $Y_t = \ln(S_t)$, to apply Ito's lemma calculate the first two derivatives
\begin{align}
Y^{\prime}_t = \frac{1}{S_t}, \; Y^{\prime \prime}_t = \frac{1}{S_t^2}
\end{align}
Then via Ito's lemma
\begin{align}
dY_t &= Y^{\prime}_t dS_t - \frac{1}{2} \cdot Y^{\prime \prime}_t dS_t^2 \\
     &= \frac{1}{S_t}dS_t - \frac{1}{2} \cdot \frac{1}{S_t^2} dS_t^2 \\
     &= \frac{1}{S_t} \left(\mu S_t dt + \sigma S_t dW_t \right) - \frac{1}{2} \cdot \frac{1}{S_t^2} \cdot (\sigma S_t)^2 dt \\
     &= \mu dt + \sigma dW_t - \frac{1}{2} \sigma^2 dt \\
     &= \left(\mu- \frac{1}{2}\sigma^2 \right)dt + \sigma dW_t \\
     &= \tilde{\mu}dt + \sigma dW_t
\end{align}
The author uses $\tilde{\mu}$ throughout the paper and then with some integration the following
\begin{align}
\int_t^T dY_t &= \tilde{\mu} \int_t^T d_t + \sigma \int_t^T dB_t \\
Y_T - Y_t     &= \tilde{\mu} \cdot (T-t) + \sigma \cdot (B_T - B_t) 
\end{align}
This result is the classic solution to the geometric brownian SDE, yet to manipulate it to the past trend signal Eq(1). This can be done by simply divided by $(T-t)$. The author rewrites this in terms of normally distributed variable $U\sim N(0,1)$. This is done by using the formula differenced brownians. Therefore the trend signal can be expressed as 
\begin{align}
\frac{1}{T-t} \left(Y_T - Y_t\right) &= \tilde{\mu} + \frac{\sigma}{T-t} \cdot (B_T - B_t) \\
                                     &= \tilde{\mu} + \frac{\sigma}{T-t} \cdot U\sqrt{T-t}\\
                                     &= \tilde{\mu} + \frac{\sigma}{\sqrt{T-t}}U
\end{align}

To calculate the z-score calculate the first two moments, throughout this paper I'll write the trend signal $\tau = \frac{1}{T-t} (Y_T - Y_t)$
\begin{align}
\mathbf{E}(\tau) &= \mathbf{E}\left( \tilde{\mu} + \frac{\sigma}{\sqrt{T-t}}U \right) \\
                 &= \tilde{\mu}
\end{align}
Then for variance 
\begin{align}
\textrm{Var}(\tau) &= \textrm{Var}\left(\tilde{\mu} + \frac{\sigma}{\sqrt{T-t}}U \right) \\
                   &= \textrm{Var}\left(\frac{\sigma}{\sqrt{T-t}}U \right) \\
                   &= \frac{\sigma^2}{T-t} \\
\end{align}